In [8]:
url = "https://www.nykaa.com/search/result/?q=healthcare&root=search&searchType=Manual&sourcepage=Search+Page"

In [50]:
from selenium import webdriver
from bs4 import BeautifulSoup
import re
import pandas as pd
import time

url = "https://www.nykaa.com/search/result/?q=healthcare&root=search&searchType=Manual&sourcepage=Search+Page"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

titles = soup.find_all("h2", class_="css-xrzmfa")

print("Number of products:", len(titles))

products = []

for product in titles[:20]:

    product_name = product.get_text(strip=True)

    parent = product.parent.parent.parent
    text = parent.get_text(" ", strip=True)

    # MRP
    mrp_match = re.search(r"Regular price ₹(\d+)", text)
    mrp = mrp_match.group(1) if mrp_match else "N/A"

    # Selling Price
    price_match = re.search(r"Discounted price ₹(\d+)", text)
    price = price_match.group(1) if price_match else "N/A"

    # Discount
    discount_match = re.search(r"(\d+)% Off", text)
    discount = discount_match.group(1) + "%" if discount_match else "N/A"

    # Brand
    brand_match = re.match(r"(\S+)", product_name)
    brand = brand_match.group(1) if brand_match else "N/A"

    # Rating
    rating = "N/A"
    current = product

    for i in range(5):
        if current:
            current_text = current.get_text(" ", strip=True)
            rating_match = re.search(r"\b[0-5]\.\d\b", current_text)

            if rating_match:
                rating = rating_match.group(0)
                break

            current = current.parent

    # Reviews
    reviews_match = re.search(r"\(\s*(\d+)\s*\)", text)
    reviews = reviews_match.group(1) if reviews_match else "N/A"

    # Product URL
    link = product.find_parent("a", href=True)

    if link:
        href = link["href"]

        if href.startswith("http"):
            product_url = href
        else:
            product_url = "https://www.nykaa.com" + href
    else:
        product_url = "N/A"

    # Category
    category = "Healthcare"

    products.append({
        "Product Name": product_name,
        "Brand": brand,
        "MRP": mrp,
        "Selling Price": price,
        "Discount": discount,
        "Rating": rating,
        "Reviews": reviews,
        "Product URL": product_url,
        "Category": category
    })

driver.quit()

print("Products collected:", len(products))

Number of products: 20
Products collected: 20


In [14]:
# Convert the scraped Healthcare products into a DataFrame
# to check whether MRP and Selling Price were captured correctly.

healthcare_df = pd.DataFrame(products)

healthcare_df[[
    "Product Name",
    "MRP",
    "Selling Price",
    "Discount"
]].head(10)

,Product Name,MRP,Selling Price,Discount
0,Carbamide Forte Biotin Tablets for Hair Growth...,920,599,35%
1,Brinton Healthcare Uvdoux Face & Body Sunscree...,845,777,8%
2,Inlife Biotin Advanced Hair Skin & Nails Suppl...,619,449,27%
3,Midazzle Dual Action Plastic Tongue Cleaner Wi...,75,71,5%
4,"BODYOPS Biotin Calcium, Magnesium and Zinc 100...",N/A,N/A,N/A
5,Majestique Luxury Portable Baby Care Kit,599,497,17%
6,HealthKart Hk Vitals Skin Radiance Collagen Su...,975,909,7%
7,Biocule Vitamin C Strawberry Plump Lip Plumpin...,N/A,N/A,N/A
8,YourHappyLife Marine Collagen Reglow 8000mg wi...,2199,1803,18%
9,HealthKart HK Vitals Skin Radiance Collagen - ...,1398,1169,16%


In [51]:
df = pd.DataFrame(products)
df

,Product Name,Brand,MRP,Selling Price,Discount,Rating,Reviews,Product URL,Category
0,Carbamide Forte Biotin Tablets for Hair Growth...,Carbamide,920,599,35%,N/A,205,https://www.nykaa.com/carbamide-forte-biotinov...,Healthcare
1,Brinton Healthcare Uvdoux Face & Body Sunscree...,Brinton,845,777,8%,N/A,2507,https://www.nykaa.com/brinton-healthcare-uvdou...,Healthcare
2,Inlife Biotin Advanced Hair Skin & Nails Suppl...,Inlife,619,449,27%,N/A,43,https://www.nykaa.com/inlife-advanced-hair-ski...,Healthcare
3,Midazzle Dual Action Plastic Tongue Cleaner Wi...,Midazzle,75,71,5%,N/A,9,https://www.nykaa.com/midazzle-2ools-plastic-t...,Healthcare
4,"BODYOPS Biotin Calcium, Magnesium and Zinc 100...",BODYOPS,N/A,N/A,N/A,N/A,1,https://www.nykaa.com/bodyops-biotin-calcium-m...,Healthcare
5,Majestique Luxury Portable Baby Care Kit,Majestique,599,497,17%,N/A,104,https://www.nykaa.com/majestique-luxury-portab...,Healthcare
6,HealthKart Hk Vitals Skin Radiance Collagen Su...,HealthKart,975,909,7%,N/A,10214,https://www.nykaa.com/healthkart-hk-vitals-ski...,Healthcare
7,Biocule Vitamin C Strawberry Plump Lip Plumpin...,Biocule,N/A,N/A,N/A,N/A,18,https://www.nykaa.com/biocule-vitamin-c-strawb...,Healthcare
8,YourHappyLife Marine Collagen Reglow 8000mg wi...,YourHappyLife,2199,1803,18%,N/A,151,https://www.nykaa.com/yourhappylife-collagen-s...,Healthcare
9,HealthKart HK Vitals Skin Radiance Collagen - ...,HealthKart,1398,1169,16%,N/A,517,https://www.nykaa.com/healthkart-hk-vitals-ski...,Healthcare


In [52]:
df.to_csv("nykaa_healthcare_products.csv", index=False)

print("CSV file created successfully!")
print("Total products saved:", len(df))

CSV file created successfully!
Total products saved: 20


In [16]:
# Check the Healthcare products for which price information is missing.

healthcare_df[
    healthcare_df["MRP"] == "N/A"
][[
    "Product Name",
    "MRP",
    "Selling Price",
    "Discount",
    "Product URL"
]]

,Product Name,MRP,Selling Price,Discount,Product URL
4,"BODYOPS Biotin Calcium, Magnesium and Zinc 100...",N/A,N/A,N/A,https://www.nykaa.com/bodyops-biotin-calcium-m...
7,Biocule Vitamin C Strawberry Plump Lip Plumpin...,N/A,N/A,N/A,https://www.nykaa.com/biocule-vitamin-c-strawb...
12,Miduty Marine Collagen 4X Hydrolysed Collagen ...,N/A,N/A,N/A,https://www.nykaa.com/miduty-marine-collagen-4...


In [17]:
### Step-3.1 Re-check Missing Healthcare Prices

# Some Healthcare products did not show price information in the listing page.
# Open their individual product pages and try to extract the actual prices.

missing_products = healthcare_df[
    healthcare_df["MRP"] == "N/A"
]

for index, row in missing_products.iterrows():

    product_url = row["Product URL"]

    driver = webdriver.Chrome()
    driver.get(product_url)

    time.sleep(4)

    product_soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )

    page_text = product_soup.get_text(" ", strip=True)

    print("\nProduct:", row["Product Name"])
    print("URL:", product_url)

    # Find all rupee amounts from the product page
    prices = re.findall(r"₹\s*([\d,]+)", page_text)

    print("Prices found:", prices)

    driver.quit()


Product: BODYOPS Biotin Calcium, Magnesium and Zinc 1000mcg Capsules
URL: https://www.nykaa.com/bodyops-biotin-calcium-magnesium-and-zinc-1000mcg-capsules/p/18860612?productId=18860612&pps=5
Prices found: ['998', '998', '300', '300', '2', '998', '998']

Product: Biocule Vitamin C Strawberry Plump Lip Plumping Cream
URL: https://www.nykaa.com/biocule-vitamin-c-strawberry-plump-lip-plumping-cream/p/8261562?productId=8261562&pps=8
Prices found: ['299', '299', '300', '300', '2', '299', '299']

Product: Miduty Marine Collagen 4X Hydrolysed Collagen Peptides Skin ...
URL: https://www.nykaa.com/miduty-marine-collagen-4x/p/20200245?productId=20200245&pps=13
Prices found: ['1960', '1960', '300', '300', '2', '1960', '1960']


In [18]:
### Step-3.2 Update Missing Healthcare Prices

# Add the actual MRP and Selling Price found from the individual product pages.

healthcare_df.loc[
    healthcare_df["Product Name"].str.startswith("BODYOPS"),
    ["MRP", "Selling Price", "Discount"]
] = [998, 998, "0%"]

healthcare_df.loc[
    healthcare_df["Product Name"].str.startswith("Biocule"),
    ["MRP", "Selling Price", "Discount"]
] = [299, 299, "0%"]

healthcare_df.loc[
    healthcare_df["Product Name"].str.startswith("Miduty"),
    ["MRP", "Selling Price", "Discount"]
] = [1960, 1960, "0%"]

In [19]:
# Verify that the Healthcare price information is now complete.

healthcare_df[[
    "Product Name",
    "MRP",
    "Selling Price",
    "Discount"
]]

,Product Name,MRP,Selling Price,Discount
0,Carbamide Forte Biotin Tablets for Hair Growth...,920,599,35%
1,Brinton Healthcare Uvdoux Face & Body Sunscree...,845,777,8%
2,Inlife Biotin Advanced Hair Skin & Nails Suppl...,619,449,27%
3,Midazzle Dual Action Plastic Tongue Cleaner Wi...,75,71,5%
4,"BODYOPS Biotin Calcium, Magnesium and Zinc 100...",998,998,0%
5,Majestique Luxury Portable Baby Care Kit,599,497,17%
6,HealthKart Hk Vitals Skin Radiance Collagen Su...,975,909,7%
7,Biocule Vitamin C Strawberry Plump Lip Plumpin...,299,299,0%
8,YourHappyLife Marine Collagen Reglow 8000mg wi...,2199,1803,18%
9,HealthKart HK Vitals Skin Radiance Collagen - ...,1398,1169,16%


In [21]:
# Load the cleaned master dataset

import pandas as pd

master_df = pd.read_csv("nykaa_master_cleaned.csv")

print("Master dataset loaded successfully!")
print("Total products:", len(master_df))

Master dataset loaded successfully!
Total products: 60


In [23]:
# Remove duplicate product names before creating the mapping
# This prevents the InvalidIndexError during the update

healthcare_mapping = healthcare_df.drop_duplicates(
    subset="Product Name"
).set_index("Product Name")

healthcare_cols = [
    "Brand",
    "MRP",
    "Selling Price",
    "Discount",
    "Reviews",
    "Product URL"
]

mask = master_df["Category"] == "Healthcare"

for col in healthcare_cols:
    master_df.loc[mask, col] = (
        master_df.loc[mask, "Product Name"].map(
            healthcare_mapping[col]
        )
    )

print("Healthcare data updated successfully!")

Healthcare data updated successfully!


C:\Users\VELIKINTI AKSHAYA\AppData\Local\Temp\ipykernel_18532\4044051227.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan '845' '619' '75' nan '599' 299 '899' nan '509' nan nan nan nan '899'
 nan nan nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  master_df.loc[mask, col] = (
C:\Users\VELIKINTI AKSHAYA\AppData\Local\Temp\ipykernel_18532\4044051227.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan '777' '449' '71' nan '497' 299 '729' nan '499' nan nan nan nan '729'
 nan nan nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  master_df.loc[mask, col] = (
C:\Users\VELIKINTI AKSHAYA\AppData\Local\Temp\ipykernel_18532\4044051227.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated a

In [24]:
# Convert Healthcare price and discount columns to numeric values

healthcare_df["MRP"] = pd.to_numeric(
    healthcare_df["MRP"], errors="coerce"
)

healthcare_df["Selling Price"] = pd.to_numeric(
    healthcare_df["Selling Price"], errors="coerce"
)

healthcare_df["Discount"] = (
    healthcare_df["Discount"]
    .astype("string")
    .str.replace("%", "", regex=False)
)

healthcare_df["Discount"] = pd.to_numeric(
    healthcare_df["Discount"], errors="coerce"
)

healthcare_df["Reviews"] = pd.to_numeric(
    healthcare_df["Reviews"], errors="coerce"
)

print("Healthcare data types converted successfully!")

Healthcare data types converted successfully!


In [25]:
# Check Healthcare column data types

healthcare_df[
    ["MRP", "Selling Price", "Discount", "Reviews"]
].dtypes

MRP              int64
Selling Price    int64
Discount         Int64
Reviews          int64
dtype: object

In [26]:
# Update the Healthcare information in the master dataset

healthcare_mapping = (
    healthcare_df
    .drop_duplicates(subset="Product Name")
    .set_index("Product Name")
)

healthcare_cols = [
    "Brand",
    "MRP",
    "Selling Price",
    "Discount",
    "Reviews",
    "Product URL"
]

mask = master_df["Category"] == "Healthcare"

for col in healthcare_cols:
    master_df.loc[mask, col] = (
        master_df.loc[mask, "Product Name"]
        .map(healthcare_mapping[col])
    )

print("Healthcare data updated successfully!")

Healthcare data updated successfully!


In [27]:
# Check missing values in Healthcare data

master_df[master_df["Category"] == "Healthcare"][
    ["MRP", "Selling Price", "Discount", "Reviews", "Product URL"]
].isnull().sum()

MRP              12
Selling Price    12
Discount         12
Reviews          12
Product URL      12
dtype: int64

In [30]:
# Check whether the 12 unmatched products have similar names
# regex=False treats the product name as normal text

for name in unmatched["Product Name"]:
    
    matches = healthcare_df[
        healthcare_df["Product Name"].str.contains(
            str(name)[:30],
            case=False,
            na=False,
            regex=False
        )
    ]
    
    print("\nMaster Product:", name)
    print("Possible match:")
    print(matches["Product Name"].tolist())


Master Product: Neurozan Health Supplements (23 Micronutrients Including Bot...
Possible match:
[]

Master Product: Fast&Up Vitalize Multivitamin Supplements - Orange (Tube Of ...
Possible match:
[]

Master Product: Wellman Health Supplements UK's No.1 Multivitamin( With Gins...
Possible match:
[]

Master Product: HealthKart HK Vitals Advanced Daily Multivitamin For Active ...
Possible match:
[]

Master Product: HealthKart Hk Vitals Iron And Folic Acid Supplement, Support...
Possible match:
[]

Master Product: Wellwoman 50+ Health Supplements UK's No.1 Vitamin for Women...
Possible match:
[]

Master Product: HealthKart HK Vitals Dht Blocker With Biotin Tablets, Helps ...
Possible match:
[]

Master Product: Swisse Ultra Collagen Platinum Shots - 13000MG
Possible match:
[]

Master Product: Neurozan 27 Micronutrients Tablets
Possible match:
[]

Master Product: HealthKart Hk Vitals Vitamin D3 (2000 Iu) Capsules
Possible match:
[]

Master Product: MuscleBlaze MB-VITE Multivitamin Tablets
P

In [31]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for the Neurozan product on Nykaa

url = "https://www.nykaa.com/search/result/?q=Neurozan%20Health%20Supplements"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

# Find product links
links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)
    
    if "Neurozan" in text:
        print("Product:", text[:150])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()

Product: Neurozan
URL: /brands/neurozan/c/18148?ptype=lst&id=18148&root=brand_menu,brand_list,Neurozan
--------------------------------------------------------------------------------
Product: Neurozan Health Supplements (23 Micronutrients Including Bot... ₹726 ₹581 20% Off Regular price ₹726. Discounted price ₹581. 20% Off. ( 11 )
URL: /neurozan-health-supplements-23-micronutrients-including-botanical-extract-of-ginkgo-bilobo/p/1406707?productId=1406707&pps=1
--------------------------------------------------------------------------------
Product: Neurozan 27 Micronutrients Tablets ₹1312 ₹918 30% Off Regular price ₹1312. Discounted price ₹918. 30% Off. ( 3 )
URL: /neurozan-27-micronutrients-tablets/p/21123774?productId=21123774&pps=2
--------------------------------------------------------------------------------


In [32]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for Fast&Up Vitalize on Nykaa

url = "https://www.nykaa.com/search/result/?q=Fast%26Up%20Vitalize"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)

    if "Fast&Up" in text or "Vitalize" in text:
        print("Product:", text[:200])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()

Product: Fast&Up
URL: /brands/fast-and-up/c/4994?ptype=lst&id=4994&root=brand_menu,brand_list,Fast&Up
--------------------------------------------------------------------------------
Product: Fast&Up Vitalize Multivitamin Supplements - Orange (Tube Of ... ₹290 ₹219 24% Off Regular price ₹290. Discounted price ₹219. 24% Off. ( 40 )
URL: /fast-up-vitalize-multivitamin-supplements-orange-tube-of-20/p/338412?productId=338412&pps=1
--------------------------------------------------------------------------------
Product: Fast&Up Vitalize Multivitamin Natural Beetroot Supplements -... ₹870 ₹618 29% Off Regular price ₹870. Discounted price ₹618. 29% Off. ( 49 )
URL: /fast-up-vitalize-multivitamin-supplements-orange-tube-of-20x3/p/338411?productId=338411&pps=2
--------------------------------------------------------------------------------
Product: Fast&Up Vitalize Multivitamins Supplements - Orange (Pack of... ₹580 ₹438 24% Off Regular price ₹580. Discounted price ₹438. 24% Off. ( 80 )
URL: /f

In [33]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for Wellman Health Supplements on Nykaa

url = "https://www.nykaa.com/search/result/?q=Wellman%20Health%20Supplements"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)

    if "Wellman" in text:
        print("Product:", text[:200])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()

Product: Wellman
URL: /brands/wellman/c/18142?ptype=lst&id=18142&root=brand_menu,brand_list,Wellman
--------------------------------------------------------------------------------
Product: Wellman Health Supplements UK's No.1 Multivitamin( With Gins... ₹387 ₹310 20% Off Regular price ₹387. Discounted price ₹310. 20% Off. ( 40 )
URL: /wellman-health-supplements-21-essential-vitamins-and-minerals/p/1406693?productId=1406693&pps=1
--------------------------------------------------------------------------------
Product: Wellman Health & Vitality Multivitamin Tablets For Men ₹711 ₹498 30% Off Regular price ₹711. Discounted price ₹498. 30% Off. ( 2 )
URL: /wellman-health-vitality-multivitamin-tablets-for-men/p/21123765?productId=21123765&pps=2
--------------------------------------------------------------------------------
Product: Wellman 50+ Health & Vitality Multivitamin Tablets For Men ₹729 ₹510 30% Off Regular price ₹729. Discounted price ₹510. 30% Off. ( 2 )
URL: /wellman-50-health-vi

In [34]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for HealthKart HK Vitals Advanced Daily Multivitamin on Nykaa

url = "https://www.nykaa.com/search/result/?q=HealthKart%20HK%20Vitals%20Advanced%20Daily%20Multivitamin"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)

    if "Advanced Daily Multivitamin" in text:
        print("Product:", text[:250])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()


Product: HealthKart HK Vitals Advanced Daily Multivitamin For Active ... ₹389 ₹359 8% Off Regular price ₹389. Discounted price ₹359. 8% Off. ( 547 ) 2 sizes
URL: /healthkart-hk-vitals-advanced-daily-multivitamin-for-active-women/p/12831113?productId=12831113&pps=3
--------------------------------------------------------------------------------


In [35]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for HealthKart HK Vitals Iron and Folic Acid on Nykaa

url = "https://www.nykaa.com/search/result/?q=HealthKart%20HK%20Vitals%20Iron%20Folic%20Acid"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)

    if "Iron" in text and "Folic" in text:
        print("Product:", text[:250])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()

Product: HealthKart Hk Vitals Iron And Folic Acid Supplement, Support... ₹419 ₹379 10% Off Regular price ₹419. Discounted price ₹379. 10% Off. ( 522 ) 2 sizes
URL: /healthkart-hk-vitals-iron-and-folic-acid-supplement-supports-blood-building-immunity-and-energy/p/6783534?productId=6783534&pps=1
--------------------------------------------------------------------------------
Product: Sheneed Folic Acid & Iron Supplement For Pregnancy ₹600 ₹528 12% Off Regular price ₹600. Discounted price ₹528. 12% Off. ( 144 ) 2 sizes
URL: /sheneed-folic-acid-iron-supplement-for-pregnancy/p/13473765?productId=13473765&pps=3
--------------------------------------------------------------------------------


In [36]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for Wellwoman 50+ Health Supplements on Nykaa

url = "https://www.nykaa.com/search/result/?q=Wellwoman%2050%2B%20Health%20Supplements"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)

    if "Wellwoman" in text:
        print("Product:", text[:250])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()

Product: Wellwoman
URL: /brands/wellwoman/c/18143?ptype=lst&id=18143&root=brand_menu,brand_list,Wellwoman
--------------------------------------------------------------------------------
Product: Wellwoman 50+ Health Supplements UK's No.1 Vitamin for Women... ₹573 ₹458 20% Off Regular price ₹573. Discounted price ₹458. 20% Off. ( 68 )
URL: /wellwoman-50-health-supplements-26-vitamins-and-minerals/p/1406698?productId=1406698&pps=1
--------------------------------------------------------------------------------
Product: Wellwoman 50+ Health & Vitality Multivitamin Tablets For Wom... ₹1041 ₹729 30% Off Regular price ₹1041. Discounted price ₹729. 30% Off. ( 6 )
URL: /wellwoman-50-health-vitality-multivitamin-tablets-for-women/p/21123770?productId=21123770&pps=2
--------------------------------------------------------------------------------
Product: Wellwoman Supplements UK's No.1 Multivitamin for Women (Even... ₹669 ₹535 20% Off Regular price ₹669. Discounted price ₹535. 20% Off. ( 175 )


In [37]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for HealthKart HK Vitals DHT Blocker with Biotin on Nykaa

url = "https://www.nykaa.com/search/result/?q=HealthKart%20HK%20Vitals%20DHT%20Blocker%20Biotin"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)

    if "DHT" in text and "Biotin" in text:
        print("Product:", text[:250])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()

Product: Wellbeing Nutrition Melts Hairfall Control, Biotin, DHT Bloc... Price ₹649. ₹649 ( 147 )
URL: /wellbeing-nutrition-plant-based-melts-hair-fall-control-tropical-strawberry-flavour/p/6795997?productId=6795997&pps=5
--------------------------------------------------------------------------------
Product: HealthKart HK Vitals Biotin & DHT Blocker with Biotin, Suppl... ₹1008 ₹749 26% Off Regular price ₹1008. Discounted price ₹749. 26% Off. ( 10 )
URL: /healthkart-hk-vitals-biotin-tablets-hk-vitals-dht-blocker-with-biotin-tablets/p/7133741?productId=7133741&pps=10
--------------------------------------------------------------------------------
Product: YourHappyLife Hair With Keranat™, Biotin, DHT Blockers - Dua... ₹1299 ₹1130 13% Off Regular price ₹1299. Discounted price ₹1130. 13% Off. ( 92 )
URL: /yourhappylife-capsules-for-hair-growth/p/13361035?productId=13361035&pps=13
--------------------------------------------------------------------------------
Product: YourHappyLife Hair 

In [38]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for Swisse Ultra Collagen Platinum Shots on Nykaa

url = "https://www.nykaa.com/search/result/?q=Swisse%20Ultra%20Collagen%20Platinum%20Shots%2013000MG"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)

    if "Swisse" in text and "Collagen" in text:
        print("Product:", text[:250])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()


Product: Swisse Ultra Collagen Platinum Shots - 13000MG ₹3299 ₹2804 15% Off Regular price ₹3299. Discounted price ₹2804. 15% Off. ( 6 )
URL: /swisse-ultra-collagen-platinum-shots-13000mg/p/24156130?productId=24156130&pps=2
--------------------------------------------------------------------------------


In [39]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for HealthKart HK Vitals Vitamin D3 on Nykaa

url = "https://www.nykaa.com/search/result/?q=HealthKart%20HK%20Vitals%20Vitamin%20D3"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

links = soup.find_all("a", href=True)

for link in links:
    text = link.get_text(" ", strip=True)

    if "HealthKart" in text and "Vitamin D3" in text:
        print("Product:", text[:250])
        print("URL:", link["href"])
        print("-" * 80)

driver.quit()

Product: HealthKart Hk Vitals Vitamin D3 (2000 Iu) Capsules ₹849 ₹579 32% Off Regular price ₹849. Discounted price ₹579. 32% Off. ( 409 ) 2 sizes
URL: /healthkart-hk-vitals-vitamin-d3-2000-iu-capsules/p/6783503?productId=6783503&pps=1
--------------------------------------------------------------------------------
Product: HealthKart HK Vitals Calcium + Vitamin D3 Supplement Tablets... ₹399 ₹329 18% Off Regular price ₹399. Discounted price ₹329. 18% Off. ( 195 )
URL: /healthkart-hk-vitals-calcium-vitamin-d3-supplement-tablets/p/10040523?productId=10040523&pps=3
--------------------------------------------------------------------------------
Product: HealthKart HK Vitals Calcium + Vitamin D3 Supplement With Ma... ₹309 ₹269 13% Off Regular price ₹309. Discounted price ₹269. 13% Off. ( 96 )
URL: /healthkart-calcium-tablets-for-men-and-women/p/1411716?productId=1411716&pps=9
--------------------------------------------------------------------------------
Product: HealthKart HK Vitals Vitam

In [40]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search the last 2 missing Healthcare products

searches = [
    "MuscleBlaze MB-VITE Multivitamin Tablets",
    "Dr. Morepen Biotin"
]

driver = webdriver.Chrome()

for query in searches:

    print("\n" + "=" * 100)
    print("SEARCH:", query)
    print("=" * 100)

    url = "https://www.nykaa.com/search/result/?q=" + query.replace(" ", "%20")

    driver.get(url)
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    titles = soup.find_all("h2", class_="css-xrzmfa")

    for product in titles[:10]:

        product_name = product.get_text(" ", strip=True)

        parent = product.parent.parent.parent
        text = parent.get_text(" ", strip=True)

        print("\nProduct:", product_name)
        print("Details:", text[:300])

        link = product.find_parent("a", href=True)

        if link:
            print("URL:", link["href"])

driver.quit()


SEARCH: MuscleBlaze MB-VITE Multivitamin Tablets

Product: MuscleBlaze MB-VITE Multivitamin Tablets
Details: MuscleBlaze MB-VITE Multivitamin Tablets ₹649 ₹579 11% Off Regular price ₹649. Discounted price ₹579. 11% Off. ( 225 ) 3 sizes
URL: /muscleblaze-mb-vite-multivitamin-tablets/p/826614?productId=826614&pps=1

Product: MuscleBlaze Mb Vite Multivitamin, Unflavored
Details: MuscleBlaze Mb Vite Multivitamin, Unflavored ₹889 ₹809 9% Off Regular price ₹889. Discounted price ₹809. 9% Off. ( 13 )
URL: /muscleblaze-mb-vite-multivitamin-unflavored/p/5339867?productId=5339867&pps=2

Product: MuscleBlaze MB-Vite Daily & Strongher Women Multivitamin Tab...
Details: MuscleBlaze MB-Vite Daily & Strongher Women Multivitamin Tab... Price ₹1198. ₹1198
URL: /muscleblaze-mb-vite-daily-multivitamin-tablets-strongher-women-multivitamin-tablets/p/10328609?productId=10328609&pps=3

Product: MuscleBlaze Mb-vite Multivitamin, 30 Tablets With Omega 3 Fi...
Details: MuscleBlaze Mb-vite Multivitamin, 30 Tabl

In [41]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

# Search for Dr. Morepen Biotin on Nykaa

url = "https://www.nykaa.com/search/result/?q=Dr.%20Morepen%20Biotin"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

titles = soup.find_all("h2", class_="css-xrzmfa")

for product in titles[:10]:

    product_name = product.get_text(" ", strip=True)

    parent = product.parent.parent.parent
    text = parent.get_text(" ", strip=True)

    print("\nProduct:", product_name)
    print("Details:", text[:300])

    link = product.find_parent("a", href=True)

    if link:
        print("URL:", link["href"])

driver.quit()


Product: Dr. Morepen Biotin For Hair Growth, Glowing Skin & Healthy N...
Details: Dr. Morepen Biotin For Hair Growth, Glowing Skin & Healthy N... ₹879 ₹510 42% Off Regular price ₹879. Discounted price ₹510. 42% Off. ( 36 )
URL: /dr-morepen-biotin-for-hair-growth-glowing-skin-healthy-nails-multivitamins-natural-extracts/p/2730717?productId=2730717&pps=1

Product: Dr. Morepen Testosterone Booster Tablets For Men 60S
Details: Dr. Morepen Testosterone Booster Tablets For Men 60S ₹1099 ₹626 43% Off Regular price ₹1099. Discounted price ₹626. 43% Off. ( 6 )
URL: /dr-morepen-testosterone-booster-tablets-for-men/p/2655909?productId=2655909&pps=2

Product: Be Bodywise Biotin Hair Gummies - Zinc, Fibre, Multivitamin ...
Details: BESTSELLER Be Bodywise Biotin Hair Gummies - Zinc, Fibre, Multivitamin ... ₹999 ₹849 15% Off Regular price ₹999. Discounted price ₹849. 15% Off. ( 12318 ) 2 sizes
URL: /be-bodywise-biotin-hair-gummies-zinc-fibre-multivitamin-for-stronger-hair-nails-no-added-sugar/p/1657

In [42]:
# Show the Healthcare rows with missing values

healthcare_missing = master_df[
    (master_df["Category"] == "Healthcare") &
    (master_df["MRP"].isna())
]

print(healthcare_missing[["Product Name"]])

                                         Product Name
20  Neurozan Health Supplements (23 Micronutrients...
24  Fast&Up Vitalize Multivitamin Supplements - Or...
28  Wellman Health Supplements UK's No.1 Multivita...
30  HealthKart HK Vitals Advanced Daily Multivitam...
31  HealthKart Hk Vitals Iron And Folic Acid Suppl...
32  Wellwoman 50+ Health Supplements UK's No.1 Vit...
33  HealthKart HK Vitals Dht Blocker With Biotin T...
35     Swisse Ultra Collagen Platinum Shots - 13000MG
36                 Neurozan 27 Micronutrients Tablets
37  HealthKart Hk Vitals Vitamin D3 (2000 Iu) Caps...
38           MuscleBlaze MB-VITE Multivitamin Tablets
39  Dr. Morepen Biotin For Hair Growth, Glowing Sk...


In [43]:
# Update the 12 missing Healthcare products with the values collected from Nykaa

updates = {
    20: [726, 581, 20, 11,
         "https://www.nykaa.com/neurozan-health-supplements-23-micronutrients-including-botanical-extract-of-ginkgo-bilobo/p/1406707?productId=1406707&pps=1"],

    24: [290, 219, 24, 40,
         "https://www.nykaa.com/fast-up-vitalize-multivitamin-supplements-orange-tube-of-20/p/338412?productId=338412&pps=1"],

    28: [387, 310, 20, 40,
         "https://www.nykaa.com/wellman-health-supplements-21-essential-vitamins-and-minerals/p/1406693?productId=1406693&pps=1"],

    30: [389, 359, 8, 547,
         "https://www.nykaa.com/healthkart-hk-vitals-advanced-daily-multivitamin-for-active-women/p/12831113?productId=12831113&pps=3"],

    31: [419, 379, 10, 522,
         "https://www.nykaa.com/healthkart-hk-vitals-iron-and-folic-acid-supplement-supports-blood-building-immunity-and-energy/p/6783534?productId=6783534&pps=1"],

    32: [573, 458, 20, 68,
         "https://www.nykaa.com/wellwoman-50-health-supplements-26-vitamins-and-minerals/p/1406698?productId=1406698&pps=1"],

    33: [1008, 749, 26, 10,
         "https://www.nykaa.com/healthkart-hk-vitals-biotin-tablets-hk-vitals-dht-blocker-with-biotin-tablets/p/7133741?productId=7133741&pps=10"],

    35: [3299, 2804, 15, 6,
         "https://www.nykaa.com/swisse-ultra-collagen-platinum-shots-13000mg/p/24156130?productId=24156130&pps=2"],

    36: [1312, 918, 30, 3,
         "https://www.nykaa.com/neurozan-27-micronutrients-tablets/p/21123774?productId=21123774&pps=2"],

    37: [849, 579, 32, 409,
         "https://www.nykaa.com/healthkart-hk-vitals-vitamin-d3-2000-iu-capsules/p/6783503?productId=6783503&pps=1"],

    38: [649, 579, 11, 225,
         "https://www.nykaa.com/muscleblaze-mb-vite-multivitamin-tablets/p/826614?productId=826614&pps=1"],

    39: [879, 510, 42, 36,
         "https://www.nykaa.com/dr-morepen-biotin-for-hair-growth-glowing-skin-healthy-nails-multivitamins-natural-extracts/p/2730717?productId=2730717&pps=1"]
}

for row, values in updates.items():
    master_df.loc[row, "MRP"] = values[0]
    master_df.loc[row, "Selling Price"] = values[1]
    master_df.loc[row, "Discount"] = values[2]
    master_df.loc[row, "Reviews"] = values[3]
    master_df.loc[row, "Product URL"] = values[4]

print("12 missing Healthcare products updated successfully!")

12 missing Healthcare products updated successfully!


In [44]:
# Check whether any Healthcare values are still missing

master_df[master_df["Category"] == "Healthcare"][
    ["Product Name", "MRP", "Selling Price", "Discount", "Reviews", "Product URL"]
].isnull().sum()

Product Name     0
MRP              0
Selling Price    0
Discount         0
Reviews          0
Product URL      0
dtype: int64

In [45]:
# Check the complete cleaned dataset

print("Total products:", len(master_df))
print("\nMissing values:")
print(master_df.isnull().sum())

Total products: 60

Missing values:
Product Name      0
Brand            12
MRP              10
Selling Price    10
Discount         10
Rating           40
Reviews           0
Product URL       0
Category          0
dtype: int64


In [46]:
# Check the products where Brand is still missing

master_df[master_df["Brand"].isna()][
    ["Product Name", "Category", "MRP", "Selling Price"]
]

,Product Name,Category,MRP,Selling Price
20,Neurozan Health Supplements (23 Micronutrients...,Healthcare,726,581
24,Fast&Up Vitalize Multivitamin Supplements - Or...,Healthcare,290,219
28,Wellman Health Supplements UK's No.1 Multivita...,Healthcare,387,310
30,HealthKart HK Vitals Advanced Daily Multivitam...,Healthcare,389,359
31,HealthKart Hk Vitals Iron And Folic Acid Suppl...,Healthcare,419,379
32,Wellwoman 50+ Health Supplements UK's No.1 Vit...,Healthcare,573,458
33,HealthKart HK Vitals Dht Blocker With Biotin T...,Healthcare,1008,749
35,Swisse Ultra Collagen Platinum Shots - 13000MG,Healthcare,3299,2804
36,Neurozan 27 Micronutrients Tablets,Healthcare,1312,918
37,HealthKart Hk Vitals Vitamin D3 (2000 Iu) Caps...,Healthcare,849,579


In [47]:
# Fill the missing Brand values for Healthcare products

brand_updates = {
    20: "Neurozan",
    24: "Fast&Up",
    28: "Wellman",
    30: "HealthKart",
    31: "HealthKart",
    32: "Wellwoman",
    33: "HealthKart",
    35: "Swisse",
    36: "Neurozan",
    37: "HealthKart",
    38: "MuscleBlaze",
    39: "Dr. Morepen"
}

for row, brand in brand_updates.items():
    master_df.loc[row, "Brand"] = brand

print("Missing Healthcare brands updated successfully!")

Missing Healthcare brands updated successfully!


In [48]:
# Check remaining missing values in Brand

print("Missing Brand values:", master_df["Brand"].isna().sum())

Missing Brand values: 0


In [49]:
# Check products where MRP, Selling Price, or Discount is missing

missing_prices = master_df[
    master_df["MRP"].isna() |
    master_df["Selling Price"].isna() |
    master_df["Discount"].isna()
]

print("Products with missing price information:", len(missing_prices))

missing_prices[
    ["Product Name", "Brand", "Category", "MRP", "Selling Price", "Discount"]
]

Products with missing price information: 10


,Product Name,Brand,Category,MRP,Selling Price,Discount
5,Beauty of Joseon Relief Sun Aqua-Fresh Add SPF...,Beauty,Beauty,NaN,NaN,NaN
8,Miduty Pigment Clear Glutathione Hyaluronic Ac...,Miduty,Beauty,NaN,NaN,NaN
10,The Face Shop Real Nature Face Mask,The,Beauty,NaN,NaN,NaN
12,Beauty of Joseon Relief Sunscreen Rice + Probi...,Beauty,Beauty,NaN,NaN,NaN
16,Rare Beauty Soft Pinch Matte Liquid Blush,Rare,Beauty,NaN,NaN,NaN
44,Forest Essentials Baby Body Massage Oil Dasapu...,Forest,Baby,NaN,NaN,NaN
53,Bioderma Ultra-Soothing Foaming Gel Atoderm In...,Bioderma,Baby,NaN,NaN,NaN
54,"Sebamed Baby Protective Facial Cream, pH 5.5, ...",Sebamed,Baby,NaN,NaN,NaN
56,Forest Essentials Baby Head Massage Oil Dasapu...,Forest,Baby,NaN,NaN,NaN
59,Cetaphil Restoraderm Body Moisturizer Repairs ...,Cetaphil,Baby,NaN,NaN,NaN
